# 02 — Train YOLOv8s on FLIR Thermal Dataset

In [ ]:
# Cell 1 — Install ultralytics
%pip install ultralytics -q

In [ ]:
# Cell 2 — Config (edit only this cell)
import torch
from ultralytics import YOLO

MODEL   = "yolov8s.pt"                        # YOLOv8 small — downloads automatically
DATA    = "../flir-data-set-27/data.yaml"      # path to Roboflow download data.yaml
EPOCHS  = 45
BATCH   = 8
IMGSZ   = 640
DEVICE  = 0 if torch.cuda.is_available() else "cpu"   # GPU if available, else CPU
PROJECT = "../runs/train"
NAME    = "night_vision_v1"

print(f"Using device: {'GPU (cuda:' + str(DEVICE) + ')' if isinstance(DEVICE, int) else 'CPU'}")
if isinstance(DEVICE, int):
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

In [ ]:
# Cell 3 — Load model and train
# YOLOv8 downloads yolov8s.pt automatically on first run
model = YOLO(MODEL)

results = model.train(
    data    = DATA,
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    project = PROJECT,
    name    = NAME,
)

print("\nTraining complete!")
print(f"Run saved to: {results.save_dir}")

In [ ]:
# Cell 4 — Copy best.pt to models/best.pt
import os
import shutil

best_src = os.path.join(str(results.save_dir), "weights", "best.pt")
best_dst = "../models/best.pt"

os.makedirs("../models", exist_ok=True)
shutil.copy2(best_src, best_dst)

print(f"Copied: {best_src}")
print(f"    ->  {os.path.abspath(best_dst)}")

In [ ]:
# Cell 5 — Display results.png (YOLOv8 auto-saves loss curves + metrics here)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_png = os.path.join(str(results.save_dir), "results.png")

img = mpimg.imread(results_png)
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Training Results", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Print final mAP@0.5 from training results
# results.results_dict contains the final epoch metrics
metrics = results.results_dict

print("=" * 40)
print("  Final Training Metrics")
print("=" * 40)
print(f"  mAP@0.5      : {metrics.get('metrics/mAP50(B)',    0):.4f}")
print(f"  mAP@0.5:0.95 : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision     : {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall        : {metrics.get('metrics/recall(B)',    0):.4f}")
print("=" * 40)
print(f"  Best weights  : {os.path.abspath(best_dst)}")